[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/04_lora.ipynb)

# LoRA

> **Fine-tuning series — 04 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.


## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos días."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' — a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## 2. LoRA  ·  *how-axis*

Freeze the base, train tiny low-rank adapters (`<1%` of params). Pass `peft_config`
and `SFTTrainer` wires up PEFT for you.

In [ ]:
from peft import LoraConfig

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)

trainer = SFTTrainer(
    model=model, train_dataset=sft_ds, peft_config=lora_cfg,
    args=SFTConfig(output_dir="ft_lora", dataset_text_field="text",
                   per_device_train_batch_size=2, max_steps=20,
                   learning_rate=2e-4, logging_steps=5,
                   bf16=BF16_OK, report_to="none"),
)
trainer.train()
trainer.model.print_trainable_parameters()   # confirm <1%, not 0%
del trainer, model; cleanup()

### Choosing the rank `r` — how much capacity do you need?

You don't *compute* the right rank — you start with a default and let **validation** move
it. Think of `r` as a **capacity dial**: higher `r` = more trainable params = more capacity,
but more memory and more overfitting risk; lower `r` = cheaper and harder to overfit. LoRA's
whole premise is that adaptation is *low-rank*, so you need far less than you'd guess — most
tasks live in **r = 8–32**, and doubling `r` rarely doubles quality.

**Where to start** (a starting point, not a final answer):

| Task | Start at | α (≈ 2r) |
|---|---|---|
| Style / tone / format tweak, small data | r = 4–8 | 8–16 |
| Typical instruction tuning / domain adaptation | r = 8–16 | 16–32 |
| Harder task, lots of new behavior, more data | r = 32–64 | 64–128 |
| Big domain shift + large data | r = 64–128 (or consider full FT) | … |

**The procedure (this is the real answer):**
1. Start **r = 16, α = 32**.
2. Train, watch the **validation** metric.
3. **Underfitting** (not learning enough)? → raise `r` (16→32→64) and/or target more modules.
4. **Overfitting** (train good, val worse)? → lower `r`, add data, add dropout.
5. Keep the **smallest `r`** that makes validation happy — cheaper and generalizes better.

Two things besides `r` move the parameter count: **which modules** you adapt and the **model
size**. The calculator below makes both concrete — edit the dims to your model and watch the
numbers.

In [ ]:
def lora_trainable(d_model, n_layers, kv_dim, intermediate, r, modules):
    """Trainable LoRA params = r*(out+in) per adapted matrix, summed over layers."""
    shapes = {
        "q_proj": (d_model, d_model),       "o_proj": (d_model, d_model),
        "k_proj": (kv_dim, d_model),        "v_proj": (kv_dim, d_model),   # smaller under GQA
        "gate_proj": (intermediate, d_model), "up_proj": (intermediate, d_model),
        "down_proj": (d_model, intermediate),
    }
    per_layer = sum(r * (out + inp) for m, (out, inp) in shapes.items() if m in modules)
    return per_layer * n_layers

# --- your model's dims (defaults = Qwen2.5-0.5B; change to yours) ---
d_model, n_layers, kv_dim, intermediate, total = 896, 24, 128, 4864, 494_000_000

sets = {
    "q,v only":              {"q_proj", "v_proj"},
    "all attention":         {"q_proj", "k_proj", "v_proj", "o_proj"},
    "all linear (attn+MLP)": {"q_proj", "k_proj", "v_proj", "o_proj",
                              "gate_proj", "up_proj", "down_proj"},
}

print(f"{'modules':24s} {'r':>4s} {'trainable':>14s} {'% of model':>11s}")
for label, mods in sets.items():
    for r in (8, 16, 64):
        tp = lora_trainable(d_model, n_layers, kv_dim, intermediate, r, mods)
        print(f"{label:24s} {r:4d} {tp:14,d} {tp/total:11.3%}")

print("\nRead it: more modules and higher r both cost more, with diminishing returns.")
print("Start small (q,v or all-linear at r=8-16); only climb if validation says so.")